In [1]:
import openai
import instructor
from pydantic import BaseModel,Field

In [2]:
client=instructor.from_openai(openai.OpenAI())

In [3]:
class RAGGenerationResponse(BaseModel):
    answer:str=Field(description="The Answer of the Question")

In [4]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance,VectorParams,PointStruct
load_dotenv()






# Qdrant host: "http://qdrant:6333" inside docker-compose, "http://localhost:6333" locally.
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")

def get_embedding(text,model="text-embedding-3-small"):
    response=client.embeddings.create(
        model=model,
        input=text
    )
    
    return response.data[0].embedding

def reteriver_data(query, qdrant_client, k):
    query_embedding = get_embedding(query)

    result = qdrant_client.query_points(
        collection_name="Amazon_items_collection-00",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_rating = []
    retrieved_image_urls = []
    retrieved_prices = []

    for point in result.points:
        payload = point.payload or {}
        retrieved_context_ids.append(payload.get("parent_asin"))
        retrieved_context.append(payload.get("description", ""))
        retrieved_context_rating.append(payload.get("average_rating"))
        similarity_scores.append(point.score)
        retrieved_image_urls.append(payload.get("image_url", ""))
        retrieved_prices.append(payload.get("price"))

    return (
        retrieved_context,
        retrieved_context_ids,
        retrieved_context_rating,
        similarity_scores,
        retrieved_image_urls,
        retrieved_prices
    )


def process_context(context, ids, ratings, scores):
    formatted_context = ""

    for id, chunk, rating, score in zip(ids, context, ratings, scores):
        formatted_context += (
            f"- ID: {id}\n"
            f"  Description: {chunk}\n"
            f"  Rating: {rating}\n"
            f"  Similarity Score: {score:.4f}\n\n"
        )

    return formatted_context

def build_prompt(processed_context, question):
    prompt = f"""
You are a helpful AI shopping assistant.

Use ONLY the information provided in the retrieved product context below to answer the user's question.

Rules:
- Do not make up information.
- If the answer is not present in the context, reply:
  "I couldn't find that information in the retrieved products."
- Keep your answer concise and helpful.
- Mention product IDs when relevant.

======================
Retrieved Context:
{processed_context}
======================

User Question:
{question}

Answer:
"""

    return prompt

def generate_chat(prompt):
    response = client.chat.completions.create_with_completion(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "system",
                "content": prompt
            },
            
        ],
        temperature=0.2,
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(QDRANT_URL)

    retrieved_context = reteriver_data(
        question,
        qdrant_client,
        top_k
    )
   
    processed_context = process_context(
        retrieved_context[0],
        retrieved_context[1],
        retrieved_context[2],
        retrieved_context[3]
    )

    prompt = build_prompt(
        processed_context,
        question
    )
    
    answer = generate_chat(prompt)
    final_result={
        "detemodel":answer,
        "answer":answer,
        "Question":question,
        "reterived_context_ids":retrieved_context[1],
        "reterived_context":retrieved_context[0],
        "similaritry_ecore":retrieved_context[3]
    }
    return final_result

### Rag PipeLine With Grounding Context

In [5]:
class RAGUsedContext(BaseModel):
    id:str=Field(description="The ID Of the item used answer the questions")
    description:str=Field(description="Short description of the item used to answer the Question")

class RAGGenerationResponse(BaseModel):
    answer:str=Field(description="The Answer of the Question")
    refernces:list[RAGUsedContext]=Field(description="List of item used to answer the Question")

In [29]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance,VectorParams,PointStruct
load_dotenv()

client=instructor.from_openai(openai.OpenAI())






# Qdrant host: "http://qdrant:6333" inside docker-compose, "http://localhost:6333" locally.
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")

def get_embedding(text,model="text-embedding-3-small"):
    response=client.embeddings.create(
        model=model,
        input=text
    )
    
    return response.data[0].embedding

def reteriver_data(query, qdrant_client, k):
    query_embedding = get_embedding(query)

    result = qdrant_client.query_points(
        collection_name="Amazon_items_collection-00",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_rating = []
    retrieved_image_urls = []
    retrieved_prices = []

    for point in result.points:
        payload = point.payload or {}
        retrieved_context_ids.append(payload.get("parent_asin"))
        retrieved_context.append(payload.get("description", ""))
        retrieved_context_rating.append(payload.get("average_rating"))
        similarity_scores.append(point.score)
        retrieved_image_urls.append(payload.get("image", ""))
        retrieved_prices.append(payload.get("price"))

    return (
        retrieved_context,
        retrieved_context_ids,
        retrieved_context_rating,
        similarity_scores,
        retrieved_image_urls,
        retrieved_prices
    )


def process_context(context, ids, ratings, scores,image_url,prices):
    formatted_context = ""

    for id, chunk, rating, score ,image_url,prices,in zip(ids, context, ratings, scores,image_url,prices):
        formatted_context += (
            f"- ID: {id}\n"
            f"  Description: {chunk}\n"
            f"  Rating: {rating}\n"
            f"  Similarity Score: {score:.4f}\n"
            f"  Images : {image_url}\n"
            f"  Price: {prices}\n\n"

        )

    return formatted_context

def build_prompt(processed_context, question):
    prompt = f"""
You are a helpful AI shopping assistant.

Use ONLY the information provided in the retrieved product context below to answer the user's question.

Rules:
- Do not make up information.
- If the answer is not present in the context, reply:
  "I couldn't find that information in the retrieved products."
- Keep your answer concise and helpful.
- Mention product IDs when relevant.

-The short description should have the name of the item
-The answer to the question should contain detailed information about the product ans returned with detailed specification in build points.
======================
Retrieved Context:
{processed_context}
======================

User Question:
{question}

Answer:
"""

    return prompt

def generate_chat(prompt):
    response = client.chat.completions.create_with_completion(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "system",
                "content": prompt
            },
            
        ],
        temperature=0.2,
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(QDRANT_URL)

    retrieved_context = reteriver_data(
        question,
        qdrant_client,
        top_k
    )
   
    processed_context = process_context(
        retrieved_context[0],
        retrieved_context[1],
        retrieved_context[2],
        retrieved_context[3],
        retrieved_context[4],
        retrieved_context[5]
    )
    print("Processe Context here",processed_context)
    

    prompt = build_prompt(
        processed_context,
        question
    )
    
    answer = generate_chat(prompt)
    final_result={
        "original_output":answer,
        "answer":answer,
        ##"refrences":answer.references,
        "Question":question,
        "reterived_context_ids":retrieved_context[1],
        "reterived_context":retrieved_context[0],
        "similaritry_ecore":retrieved_context[3]
    }
    return final_result,

In [30]:
output=rag_pipeline("Talescope");
output

Processe Context here - ID: B0BGLRMPQD
  Description: Monocular Telescope, 10x42 Monoculars for Adults, Usogood Compact Portable Waterproof Monocular with Hand Strap, Lightweight Handheld Pocket Telescope for Bird Watching【10x42 High Definition and Comfortable Viewing】This monoculars for adults with a 42mm objective lens provide 10x magnification, which ensures that you can easily magnify the object with a stable view when observing handheld. With a large field of view of 360ft/1000yards, you can clearly see the mountains 1200 yards away.【22.5 mm Extra Large Eyepiece, More Detail, Clearer and Brighter】Equipped with an oversized 22.5 mm eyepiece, this handheld monocular offers you a comfortable viewing experience. Including the BAK4 prism, all the lenses are fully multi-layer coated to reduce light loss by 99.99%, providing you with a brighter picture.【Extremely Lightweight to Carrying Out, Multiple Carrying Options】This monocular comes with a hand strap and double lens covers, all that

({'original_output': (RAGGenerationResponse(answer="The Monocular Telescope (ID: B0BGLRMPQD) is a compact, portable, and waterproof monocular designed for adults. It features 10x magnification with a 42mm objective lens, providing a clear and stable view with a large field of view of 360ft/1000yards. The eyepiece is an extra-large 22.5 mm for more detail and brightness, equipped with BAK4 prism and fully multi-layer coated lenses to reduce light loss by 99.99%. It is lightweight, weighing less than 9 oz, and comes with a hand strap, double lens covers, and a soft case with a neck strap for easy carrying. The size is mini and handheld (5.8'' x 2.5'' x 2.1''), allowing one-finger precision focus with a double focus ring. It can focus from as close as 3 yards to infinity, suitable for bird watching, deer watching, and sporting events. It also includes a 30-day hassle-free money-back guarantee and lifetime warranty.", refernces=[RAGUsedContext(id='B0BGLRMPQD', description='Monocular Telesc

In [8]:
rag_response = output["answer"][0]

print(rag_response.answer)

TypeError: tuple indices must be integers or slices, not str